In [1]:
# ============================================================
# Notebook : 03_causal_stability.ipynb
# Purpose  : Step 3 (Causal Stability) - part 2: assess how far the
#            lag-1 cross-variable structure depends on the particular
#            set of households, WITHOUT claiming independent external
#            replication. Three checks are reported:
#              (a) A/B split  : an INTERNAL stability check (the same 23
#                  households are partitioned; no household is in both
#                  halves, but this is resampling of one sample, not an
#                  independent cohort).
#              (b) Leave-one-household-out (LOGO) resampling of the
#                  aggregate, giving a distribution of sign- and rank-
#                  stability for the recurring links.
#              (c) Parameter sensitivity of the independence-test alpha.
#            A per-category reproduction verdict (full / partial /
#            not-reproduced) is produced for the governance step.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})
GRAY = ["0.15", "0.40", "0.60", "0.75", "0.88"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
V = meta["sensor_cont"]
DEFINING = meta["category_defining_sensor"]
ABN = meta["abnormal_categories"]

MIN_ROWS, TAU_MAX, RECUR = 200, 1, 0.25


def pcmci_cross_links(g, alpha):
    """Return dict {(src,tgt): parcorr} of lag-1 cross links for one household."""
    X = g[V].astype(float).values
    sd = X.std(axis=0)
    keep = sd > 1e-8
    names = [v for v, k in zip(V, keep) if k]
    Xs = (X[:, keep] - X[:, keep].mean(0)) / X[:, keep].std(0)
    res = PCMCI(dataframe=pp.DataFrame(Xs, var_names=names),
                cond_ind_test=ParCorr(), verbosity=0).run_pcmciplus(
                    tau_min=0, tau_max=TAU_MAX, pc_alpha=alpha)
    graph, vm = res["graph"], res["val_matrix"]
    out = {}
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and graph[i, j, 1] == "-->":
                out[(names[i], names[j])] = vm[i, j, 1]
    return out


# ---- Per-household cross links (alpha = 0.01), cached to disk for fast reruns ----
used_hh = [hh for hh in sorted(df["HH_ID"].unique())
           if len(df[df["HH_ID"] == hh]) >= MIN_ROWS]


def build_cache(alpha):
    fp = DATA / f"_pcmci_cache_a{int(alpha*1000):03d}.pkl"
    if fp.exists():
        return pd.read_pickle(fp)
    c = {hh: pcmci_cross_links(df[df["HH_ID"] == hh].sort_values("Timestamp"), alpha)
         for hh in used_hh}
    pd.to_pickle(c, fp)
    return c


cache = build_cache(0.01)
n_used = len(used_hh)


def aggregate_all(hh_set):
    """Aggregate cached links over households -> ALL cross links (no recurrence filter)."""
    acc = {}
    for hh in hh_set:
        for k, v in cache[hh].items():
            acc.setdefault(k, []).append(v)
    rows = []
    for k, vals in acc.items():
        rows.append({"source": k[0], "target": k[1], "freq_hh": len(vals),
                     "freq_share": len(vals) / len(hh_set),
                     "mean_parcorr": float(np.mean(vals)),
                     "mean_abs": float(np.mean(np.abs(vals)))})
    d = pd.DataFrame(rows)
    return d.sort_values("mean_abs", ascending=False).reset_index(drop=True) if len(d) else d


def aggregate(hh_set):
    """Recurring links only (freq_share >= RECUR)."""
    d = aggregate_all(hh_set)
    return d[d["freq_share"] >= RECUR].reset_index(drop=True) if len(d) else d


full = aggregate(used_hh)
full_keys = list(zip(full["source"], full["target"]))

# ============================================================
# (a) A/B split: internal stability check.
#     Partition households by sorted order into two disjoint halves.
# ============================================================
A = used_hh[0::2]
B = used_hh[1::2]
aggA, aggB = aggregate(A), aggregate(B)
mA = {(r.source, r.target): r.mean_parcorr for r in aggA.itertuples()}
mB = {(r.source, r.target): r.mean_parcorr for r in aggB.itertuples()}
common = [k for k in mA if k in mB]
sign_agree = np.mean([np.sign(mA[k]) == np.sign(mB[k]) for k in common]) if common else np.nan
if len(common) >= 3:
    rho_ab, _ = spearmanr([abs(mA[k]) for k in common], [abs(mB[k]) for k in common])
else:
    rho_ab = np.nan
ab_tbl = pd.DataFrame([{"split_A_n": len(A), "split_B_n": len(B),
                        "links_A": len(aggA), "links_B": len(aggB),
                        "common_links": len(common),
                        "sign_agreement": round(float(sign_agree), 3),
                        "rank_spearman": round(float(rho_ab), 3)}])
ab_tbl.to_csv(TAB / "table_ab_split.csv", index=False, encoding="utf-8-sig")

# ============================================================
# (b) Leave-one-household-out resampling of the aggregate.
#     For each recurring full-sample link, the share of LOGO folds that
#     keep it (presence) and keep its sign (sign stability); plus the
#     Spearman rank correlation of link strengths against the full sample.
# ============================================================
logo_sign, logo_presence, logo_rho = {k: [] for k in full_keys}, {k: [] for k in full_keys}, []
full_abs = {(r.source, r.target): r.mean_abs for r in full.itertuples()}
full_sign = {(r.source, r.target): np.sign(r.mean_parcorr) for r in full.itertuples()}
for leave in used_hh:
    sub = [h for h in used_hh if h != leave]
    agg = aggregate(sub)
    msub = {(r.source, r.target): r.mean_parcorr for r in agg.itertuples()}
    asub = {(r.source, r.target): r.mean_abs for r in agg.itertuples()}
    for k in full_keys:
        present = k in msub
        logo_presence[k].append(1 if present else 0)
        logo_sign[k].append(1 if (present and np.sign(msub[k]) == full_sign[k]) else 0)
    both = [k for k in full_keys if k in asub]
    if len(both) >= 3:
        rho, _ = spearmanr([full_abs[k] for k in both], [asub[k] for k in both])
        logo_rho.append(rho)
logo_rows = []
for k in full_keys:
    logo_rows.append({"source": k[0], "target": k[1],
                      "presence_share": round(float(np.mean(logo_presence[k])), 3),
                      "sign_stability": round(float(np.mean(logo_sign[k])), 3)})
logo_tbl = pd.DataFrame(logo_rows).sort_values("presence_share", ascending=False).reset_index(drop=True)
logo_tbl.to_csv(TAB / "table_logo_stability.csv", index=False, encoding="utf-8-sig")

# ============================================================
# (c) Parameter sensitivity: alpha 0.01 vs 0.05 (re-run discovery).
# ============================================================
cache05 = build_cache(0.05)
acc05 = {}
for hh in used_hh:
    for k, v in cache05[hh].items():
        acc05.setdefault(k, []).append(v)
rec05 = {k: np.mean(v) for k, v in acc05.items() if len(v) / n_used >= RECUR}
rec01 = {(r.source, r.target): r.mean_parcorr for r in full.itertuples()}
overlap = set(rec05) & set(rec01)
param_tbl = pd.DataFrame([{"alpha_0.01_recurring": len(rec01), "alpha_0.05_recurring": len(rec05),
                           "overlap": len(overlap),
                           "sign_agreement_overlap": round(float(np.mean(
                               [np.sign(rec05[k]) == np.sign(rec01[k]) for k in overlap])) if overlap else np.nan, 3)}])
param_tbl.to_csv(TAB / "table_param_sensitivity.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Per-category reproduction verdict (feeds the governance step).
#   For HR/BR/SP : compare the top lag-1 cross-parent of the defining
#     sensor between split A and split B.
#       full           : both halves have a parent, same top variable, same sign
#       partial        : both have a parent, signs of common parents agree,
#                         but the top variable differs
#       not_reproduced : sign flips, or a parent appears in only one half
#       autoregressive : neither half yields a cross-parent (self-lag only)
#   ST : compare top label-associated sensor (sign/rank) across halves.
#   ER : direct raw signal -> not applicable.
# ============================================================
def parents_of(agg, sensor):
    sub = agg[agg["target"] == sensor].sort_values("mean_abs", ascending=False)
    return {r.source: r.mean_parcorr for r in sub.itertuples()}


def assoc_top(hh_set):
    assoc = {}
    for hh in hh_set:
        g = df[df["HH_ID"] == hh].sort_values("Timestamp")
        y = g["cat_ST"].values[1:]
        for v in V:
            x = g[v].astype(float).values[:-1]
            if x.std() > 1e-8 and y.std() > 1e-8:
                assoc.setdefault(v, []).append(np.corrcoef(x, y)[0, 1])
    m = {v: float(np.nanmean(a)) for v, a in assoc.items()}
    return sorted(m.items(), key=lambda kv: -abs(kv[1]))


aggA_all, aggB_all = aggregate_all(A), aggregate_all(B)
verdict_rows = []
for c in ABN:
    sensor = DEFINING[c]
    if sensor in V:
        pa, pb = parents_of(aggA_all, sensor), parents_of(aggB_all, sensor)
        if not pa and not pb:
            verdict, ta, tb, sa = "autoregressive_only", "(none)", "(none)", np.nan
        elif pa and pb:
            ta, tb = max(pa, key=lambda k: abs(pa[k])), max(pb, key=lambda k: abs(pb[k]))
            comm = set(pa) & set(pb)
            sa = float(np.mean([np.sign(pa[k]) == np.sign(pb[k]) for k in comm])) if comm else 0.0
            if ta == tb and np.sign(pa[ta]) == np.sign(pb[tb]):
                verdict = "full"
            elif sa == 1.0:
                verdict = "partial"
            else:
                verdict = "not_reproduced"
        else:
            ta = max(pa, key=lambda k: abs(pa[k])) if pa else "(none)"
            tb = max(pb, key=lambda k: abs(pb[k])) if pb else "(none)"
            verdict, sa = "not_reproduced", np.nan
    elif c == "ST":
        ta_l, tb_l = assoc_top(A), assoc_top(B)
        ta, tb = ta_l[0][0], tb_l[0][0]
        topk_a = [v for v, _ in ta_l[:3]]
        da, db = dict(ta_l), dict(tb_l)
        comm = set(da) & set(db)
        sa = float(np.mean([np.sign(da[k]) == np.sign(db[k]) for k in comm]))
        verdict = "full" if ta == tb else ("partial" if sa >= 0.8 else "not_reproduced")
    else:
        verdict, ta, tb, sa = "direct_signal", "Button/Shout", "Button/Shout", np.nan
    verdict_rows.append({"category": c, "reproduction": verdict,
                         "top_parent_A": ta, "top_parent_B": tb,
                         "sign_agreement": (round(sa, 3) if sa == sa else "n/a")})
verdict_tbl = pd.DataFrame(verdict_rows)
verdict_tbl.to_csv(TAB / "table_category_reproduction.csv", index=False, encoding="utf-8-sig")
pd.to_pickle({(r["category"]): r["reproduction"] for r in verdict_rows}, DATA / "reproduction.pkl")

# ============================================================
# Figure 7. A/B link-strength agreement (|partial corr|, grayscale).
# ============================================================
fig, ax = plt.subplots(figsize=(4.4, 4.2))
if common:
    xa = [abs(mA[k]) for k in common]
    xb = [abs(mB[k]) for k in common]
    ax.scatter(xa, xb, s=36, color=GRAY[1], edgecolor="black", zorder=3)
    lim = max(max(xa), max(xb)) * 1.15
    ax.plot([0, lim], [0, lim], "--", color="0.4", linewidth=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("|partial corr|  split A")
ax.set_ylabel("|partial corr|  split B")
save_fig(fig, "fig07_ab_agreement")

# ============================================================
# Figure 8. LOGO sign-stability of recurring links (grayscale bars).
# ============================================================
fig, ax = plt.subplots(figsize=(6.6, 3.6))
lab = [f"{r.source}→{r.target}" for r in logo_tbl.itertuples()]
ax.bar(range(len(logo_tbl)), logo_tbl["sign_stability"], color=GRAY[2],
       edgecolor="black", linewidth=0.5)
ax.axhline(0.9, color="black", linestyle="--", linewidth=1.0, label="0.90")
ax.set_xticks(range(len(logo_tbl)))
ax.set_xticklabels(lab, rotation=90, fontsize=6)
ax.set_ylabel("LOGO sign stability")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
save_fig(fig, "fig08_logo_sign_stability")

# ---- Console summary ----
print("=== Step 3 (part 2): causal stability ===")
print(f"Households: {n_used}")
print()
print("(a) A/B internal stability check:")
print(ab_tbl.to_string(index=False))
print()
print("(b) LOGO resampling (presence & sign stability of recurring links):")
print(logo_tbl.to_string(index=False))
print(f"    mean LOGO rank-Spearman vs full sample: {np.mean(logo_rho):.3f}")
print()
print("(c) Parameter sensitivity (alpha 0.01 vs 0.05):")
print(param_tbl.to_string(index=False))
print()
print("Per-category reproduction verdict:")
print(verdict_tbl.to_string(index=False))
print()
print("Figures saved: fig07_ab_agreement, fig08_logo_sign_stability  (png + pdf, 600 dpi)")
print("Tables saved : table_ab_split, table_logo_stability, table_param_sensitivity, table_category_reproduction")
print("Artifact     : reproduction.pkl  (per-category verdict for the governance step)")


=== Step 3 (part 2): causal stability ===
Households: 23

(a) A/B internal stability check:
 split_A_n  split_B_n  links_A  links_B  common_links  sign_agreement  rank_spearman
        12         11       14       10             6             1.0          0.543

(b) LOGO resampling (presence & sign stability of recurring links):
     source      target  presence_share  sign_stability
        CO2    Humidity           1.000           1.000
        CO2        TVOC           1.000           1.000
Illuminance Temperature           1.000           1.000
Activity_IR  SleepPhase           1.000           1.000
 SleepScore  SleepPhase           1.000           1.000
Activity_IR Temperature           0.739           0.739
        CO2 Temperature           0.739           0.739
   Humidity Temperature           0.739           0.739
       TVOC         CO2           0.739           0.739
    mean LOGO rank-Spearman vs full sample: 0.987

(c) Parameter sensitivity (alpha 0.01 vs 0.05):
 alpha_0.0